In [0]:
# 08_context_seed — Risk Card context JSONL → context schema 적재
# asset_context / assignment_routing은 Landing 원본이나 Gold Core 본문이 아니라 Risk Card 조회용 context다.

In [0]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("asset_context_path", "data/dev/context/asset_context.jsonl")
dbutils.widgets.text("assignment_routing_path", "data/dev/context/assignment_routing.jsonl")
dbutils.widgets.text("context_seed_required", "true")

CATALOG = dbutils.widgets.get("catalog")
ASSET_CONTEXT_RELATIVE_PATH = dbutils.widgets.get("asset_context_path").strip("/")
ASSIGNMENT_ROUTING_RELATIVE_PATH = dbutils.widgets.get("assignment_routing_path").strip("/")
CONTEXT_SEED_REQUIRED = dbutils.widgets.get("context_seed_required").strip().lower() == "true"
ASSET_CONTEXT_LOCAL_PATH = f"/Workspace/Shared/access-drift/files/{ASSET_CONTEXT_RELATIVE_PATH}"
ASSIGNMENT_ROUTING_LOCAL_PATH = f"/Workspace/Shared/access-drift/files/{ASSIGNMENT_ROUTING_RELATIVE_PATH}"
ASSET_CONTEXT_PATH = f"file:{ASSET_CONTEXT_LOCAL_PATH}"
ASSIGNMENT_ROUTING_PATH = f"file:{ASSIGNMENT_ROUTING_LOCAL_PATH}"


In [0]:
import os

missing = []
for label, path in (
    ("asset_context", ASSET_CONTEXT_LOCAL_PATH),
    ("assignment_routing", ASSIGNMENT_ROUTING_LOCAL_PATH),
):
    if not os.path.isfile(path):
        missing.append((label, path))
        continue
    print(f"{label} path OK: {path}")

if missing and not CONTEXT_SEED_REQUIRED:
    message = "context seed files missing; optional task skipped: " + ", ".join(path for _, path in missing)
    print(message)
    dbutils.notebook.exit(message)

if missing:
    details = "\n".join(f"{label}: {path}" for label, path in missing)
    raise RuntimeError(
        f"Context seed 파일을 찾을 수 없습니다:\n{details}\n"
        "bundle deploy 또는 workspace import가 완료됐는지 확인하세요."
    )


In [0]:
asset_df = spark.read.json(ASSET_CONTEXT_PATH).select(
    "source_asset_id",
    "data_scope_label",
    "sensitive_data_examples",
    "sensitivity_reason",
)

(asset_df.write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.context.asset_context"))

asset_count = spark.table(f"{CATALOG}.context.asset_context").count()
assert asset_count > 0, "asset_context has no rows"
print(f"asset_context: {asset_count} rows")


In [0]:
routing_df = spark.read.json(ASSIGNMENT_ROUTING_PATH).select(
    "subject_type",
    "subject_id",
    "asset_id",
    "drift_type",
    "assignee_name",
    "assignee_team",
    "routing_reason",
)

(routing_df.write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.context.assignment_routing"))

routing_count = spark.table(f"{CATALOG}.context.assignment_routing").count()
assert routing_count > 0, "assignment_routing has no rows"
print(f"assignment_routing: {routing_count} rows")


In [0]:
asset_match_df = spark.sql(f"""
  SELECT c.source_asset_id, a.asset_id AS silver_asset_id
  FROM {CATALOG}.context.asset_context c
  LEFT JOIN {CATALOG}.silver.silver_assets a
    ON c.source_asset_id = a.source_asset_id
""")
asset_unmatched = asset_match_df.where("silver_asset_id IS NULL").count()
assert asset_unmatched == 0, f"asset_context↔silver unmatched rows: {asset_unmatched}"

routing_match_df = spark.sql(f"""
  SELECT
    r.subject_type,
    r.subject_id,
    r.asset_id,
    r.drift_type,
    g.case_id
  FROM {CATALOG}.context.assignment_routing r
  LEFT JOIN {CATALOG}.eval.ground_truth_case g
    ON r.subject_type = g.subject_type
   AND r.subject_id = g.subject_id
   AND r.asset_id = g.asset_id
   AND r.drift_type = g.drift_type
""")
routing_unmatched = routing_match_df.where("case_id IS NULL").count()
assert routing_unmatched == 0, f"assignment_routing↔eval unmatched rows: {routing_unmatched}"

spark.table(f"{CATALOG}.context.asset_context").show(truncate=False)
spark.table(f"{CATALOG}.context.assignment_routing").show(truncate=False)

summary = [
    "context seed smoke test passed",
    f"asset_context: {asset_count} rows",
    f"assignment_routing: {routing_count} rows",
    f"asset_context_silver_unmatched: {asset_unmatched}",
    f"assignment_routing_eval_unmatched: {routing_unmatched}",
]
dbutils.notebook.exit("\n".join(summary))
